# 05 - Extended features + LightGBM/shrunk-mean blend

Two simultaneous upgrades on top of notebook 03:

1. **31 new features**: missingness counts and pattern, 7 engineered
   ratios (earnings_yield, accel_growth, asset_turnover, gross_to_op_gap,
   quality_composite, distress_flag, size_proxy), and cohort-relative
   z-scores for valuation / quality / growth features within both
   `(sector × start_year)` and `(archetype × start_year)`.
2. **Convex blend** between the LightGBM prediction and the
   per-archetype shrunk-mean baseline, weight `α` searched on the
   2022 holdout. The two models err in opposite directions (LightGBM
   worse on novel tickers, shrunk-mean worse on overlap-tickers), so a
   blend dominates either alone — particularly important for test, where
   100% of tickers are novel.

## How we categorise NEW tickers (the test cohort)

Every categorisation in this pipeline derives from features alone, so
it generalises to the anonymised test tickers:

| categorisation | how it works for test rows |
|---|---|
| `sector_code` | given directly in test.csv |
| `archetype` | KMeans clusterer fit on train fundamentals predicts the cluster from the test row's fundamentals |
| `(sector × year)` z-scores | computed within test's own `(sector × 2024)` cohort using features only — no labels touched |
| `(archetype × year)` z-scores | same, within test's own `(archetype × 2024)` cohort |
| `missing_pattern_hash`, `n_missing_total` | derived directly from feature presence per row |

No ticker identity is ever used. Each test row is placed into the same
category structure as train rows by virtue of its fundamentals.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)

ROOT = Path.cwd().resolve().parent
DATA_DIR = ROOT / 'data' / 'raw'
SUBMISSION_DIR = ROOT / 'submissions'
SUBMISSION_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(ROOT / 'scripts'))
from data_prep import (  # noqa: E402
    load_raw,
    build_extended_features,
    time_split,
    clip_target_train_only,
    rmse,
    ArchetypeClusterer,
    shrunk_archetype_means,
)

train, test, sample_submission = load_raw(DATA_DIR)
train.shape, test.shape

((23070, 39), (8520, 36))

## Split + leakage-safe archetype assignment

The clusterer is fit ONLY on the train fold (start_year < 2022). It is
then applied to the full train DataFrame and to test, which means every
row — including the 2022 validation rows and the 2024 test strangers —
gets an archetype assigned purely from its fundamentals.

In [2]:
split = time_split(train, valid_year=2022)
print(split.description)

clusterer = ArchetypeClusterer(k=8).fit(train.iloc[split.train_idx])
train['archetype'] = clusterer.predict(train)
test['archetype'] = clusterer.predict(test)

X_full = build_extended_features(train)
X_test = build_extended_features(test)
y_full = train['return_pct']

assert list(X_full.columns) == list(X_test.columns), \
    'train/test feature columns must match exactly'
print(f'feature count: {X_full.shape[1]} (vs 51 in v1)')

X_train = X_full.iloc[split.train_idx]
X_valid = X_full.iloc[split.valid_idx]
y_train_raw = y_full.iloc[split.train_idx]
y_valid = y_full.iloc[split.valid_idx]
y_train, lo, hi = clip_target_train_only(y_train_raw, 1.0, 99.0)
print(f'training target clipped at [{lo:.2f}, {hi:.2f}]')

train start_year<2022 -> validate start_year==2022; validation forward window ends 2023-12-31


feature count: 82 (vs 51 in v1)
training target clipped at [-81.01, 327.94]


In [3]:
# Honest sub-metric: novel-ticker subset of the 2022 fold = test analog.
train_tickers = set(train.iloc[split.train_idx]['ticker'])
valid_tickers = train.iloc[split.valid_idx]['ticker'].values
is_novel = ~pd.Series(valid_tickers).isin(train_tickers).values
valid_archetype = train.iloc[split.valid_idx]['archetype'].values

def score(name, pred):
    return {
        'model': name,
        'rmse_all': round(rmse(y_valid, pred), 3),
        'rmse_novel': round(rmse(y_valid.values[is_novel], pred[is_novel]), 3),
        'rmse_overlap': round(rmse(y_valid.values[~is_novel], pred[~is_novel]), 3),
    }

def per_arc(pred):
    return (pd.DataFrame({'a': valid_archetype, 'e2': (y_valid.values - pred) ** 2})
            .groupby('a').agg(n=('e2','size'),
                              rmse=('e2', lambda s: float(np.sqrt(s.mean()))))
            .round(2))

## LightGBM with the extended feature set

Same Huber + year-weights recipe as notebook 03, but now with 82 features
and four LightGBM categoricals (`sector_code`, `archetype`,
`distress_flag`, `missing_pattern_hash`). The two new categoricals matter
because LightGBM splits cleanly on a discrete partition rather than
searching a continuous threshold.

In [4]:
train_years = train.iloc[split.train_idx]['start_year'].values
w_train = (train_years - 2019 + 1).astype(float)
w_train = w_train / w_train.mean()

CAT_FEATURES = [c for c in ['sector_code', 'archetype', 'distress_flag',
                            'missing_pattern_hash']
                if c in X_train.columns]
for c in CAT_FEATURES:
    X_train[c] = X_train[c].astype('int32', errors='ignore')
    X_valid[c] = X_valid[c].astype('int32', errors='ignore')
    X_test[c]  = X_test[c].astype('int32',  errors='ignore')
print('categorical features:', CAT_FEATURES)

PARAMS = dict(
    objective='huber',
    alpha=100.0,
    metric='rmse',
    learning_rate=0.04,
    num_leaves=31,
    min_data_in_leaf=80,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    verbosity=-1,
    seed=42,
)

dtrain = lgb.Dataset(X_train, label=y_train, weight=w_train,
                     categorical_feature=CAT_FEATURES, free_raw_data=False)
dvalid = lgb.Dataset(X_valid, label=y_valid, reference=dtrain,
                     categorical_feature=CAT_FEATURES, free_raw_data=False)

lgb_model = lgb.train(
    PARAMS,
    dtrain,
    num_boost_round=2000,
    valid_sets=[dtrain, dvalid],
    valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(120, verbose=False),
               lgb.log_evaluation(0)],
)
print(f'best iteration: {lgb_model.best_iteration}')

lgb_pred = lgb_model.predict(X_valid)
results = [score('lgb extended features', lgb_pred)]
display(pd.DataFrame(results))

categorical features: ['sector_code', 'archetype', 'distress_flag', 'missing_pattern_hash']


best iteration: 1


,model,rmse_all,rmse_novel,rmse_overlap
0,lgb extended features,64.731,65.847,64.683


In [5]:
imp = (pd.DataFrame({
    'feature': lgb_model.feature_name(),
    'gain':    lgb_model.feature_importance('gain'),
    'split':   lgb_model.feature_importance('split'),
}).sort_values('gain', ascending=False))
print('top 25 by gain:')
display(imp.head(25))

new_feat_keywords = ['n_missing', 'missing_pattern', 'earnings_yield',
                     'accel_growth', 'asset_turnover', 'gross_to_op_gap',
                     'quality_composite', 'distress_flag', 'size_proxy',
                     'sect_yr_z', 'arc_yr_z']
new_mask = imp['feature'].str.contains('|'.join(new_feat_keywords))
print(f'\nNew-feature gain share: '
      f'{imp.loc[new_mask, "gain"].sum() / imp["gain"].sum():.1%}')

top 25 by gain:


,feature,gain,split
0,start_year,9.265900e+06,2
33,sector_code,1.582226e+06,5
1,pe_ttm,5.907276e+05,2
3,price_to_sales,4.716156e+05,5
8,roa,2.938385e+05,2
12,revenue_growth_yoy,2.121824e+05,2
17,eps_diluted,1.286598e+05,2
16,eps_basic,9.461900e+04,1
68,price_to_sales_sect_yr_z,8.942920e+04,2
69,net_margin_sect_yr_z,7.975220e+04,1



New-feature gain share: 2.5%


## Per-archetype shrunk-mean baseline

Predict each row as the train-fold mean return of its archetype, shrunk
toward the global mean with prior weight 200. This is a strong "smart
dummy" that beats `dummy_mean` on novel tickers — exactly the
complementary failure mode to LightGBM.

In [6]:
train_arc = train.iloc[split.train_idx]['archetype']
arc_shrunk = shrunk_archetype_means(train_arc, y_train, prior=200.0)
shrunk_pred = pd.Series(valid_archetype).map(arc_shrunk).values
results.append(score('archetype shrunk mean', shrunk_pred))
display(pd.DataFrame(results))

,model,rmse_all,rmse_novel,rmse_overlap
0,lgb extended features,64.731,65.847,64.683
1,archetype shrunk mean,64.767,65.670,64.728


## Convex blend search

`pred = α × lgb + (1 - α) × shrunk_mean`. Sweep α from 0 to 1 in 0.05
steps and pick the value that minimises the **novel-ticker** RMSE — that
is the metric that matches the test situation. The overall RMSE on the
2022 fold is reported as a secondary check.

In [7]:
alphas = np.arange(0.0, 1.01, 0.05)
sweep = []
for a in alphas:
    blend = a * lgb_pred + (1 - a) * shrunk_pred
    sweep.append({
        'alpha': round(float(a), 2),
        'rmse_all':    round(rmse(y_valid, blend), 4),
        'rmse_novel':  round(rmse(y_valid.values[is_novel], blend[is_novel]), 4),
        'rmse_overlap':round(rmse(y_valid.values[~is_novel], blend[~is_novel]), 4),
    })
sweep_df = pd.DataFrame(sweep)
best_novel = sweep_df.loc[sweep_df['rmse_novel'].idxmin()]
best_all   = sweep_df.loc[sweep_df['rmse_all'].idxmin()]

display(sweep_df)
print(f"\nbest alpha by novel-ticker RMSE: {best_novel['alpha']} -> "
      f"all={best_novel['rmse_all']}, novel={best_novel['rmse_novel']}")
print(f"best alpha by overall RMSE:      {best_all['alpha']} -> "
      f"all={best_all['rmse_all']}, novel={best_all['rmse_novel']}")

# We chase novel-ticker RMSE because test = 100% novel tickers.
ALPHA = float(best_novel['alpha'])
blend_pred = ALPHA * lgb_pred + (1 - ALPHA) * shrunk_pred
results.append(score(f'BLEND (alpha={ALPHA:.2f})', blend_pred))
results_df = pd.DataFrame(results).sort_values('rmse_novel')
display(results_df)

,alpha,rmse_all,rmse_novel,rmse_overlap
0,0.00,64.7669,65.6703,64.7283
1,0.05,64.7424,65.6576,64.7033
2,0.10,64.7203,65.6472,64.6807
3,0.15,64.7006,65.6391,64.6605
4,0.20,64.6833,65.6332,64.6427
5,0.25,64.6684,65.6296,64.6273
6,0.30,64.6558,65.6283,64.6142
7,0.35,64.6456,65.6292,64.6036
8,0.40,64.6379,65.6324,64.5953
9,0.45,64.6325,65.6378,64.5895



best alpha by novel-ticker RMSE: 0.3 -> all=64.6558, novel=65.6283
best alpha by overall RMSE:      0.55 -> all=64.6289, novel=65.6555


,model,rmse_all,rmse_novel,rmse_overlap
2,BLEND (alpha=0.30),64.656,65.628,64.614
1,archetype shrunk mean,64.767,65.670,64.728
0,lgb extended features,64.731,65.847,64.683


In [8]:
print('Per-archetype RMSE — LightGBM vs blend')
lgb_arc = per_arc(lgb_pred).rename(columns={'rmse':'lgb'})
blend_arc = per_arc(blend_pred).rename(columns={'rmse':'blend'})
compare = lgb_arc.join(blend_arc['blend'])
compare['blend_minus_lgb'] = (compare['blend'] - compare['lgb']).round(2)
compare['share_of_sq_err'] = (
    (compare['blend'] ** 2 * compare['n']) /
    (compare['blend'] ** 2 * compare['n']).sum() * 100
).round(1)
display(compare.sort_values('share_of_sq_err', ascending=False))

Per-archetype RMSE — LightGBM vs blend


,n,lgb,blend,blend_minus_lgb,share_of_sq_err
a,,,,,
0,747,122.37,122.20,-0.17,40.2
5,1354,63.20,63.32,0.12,19.6
1,615,89.98,90.42,0.44,18.1
2,949,45.49,45.36,-0.13,7.0
3,731,49.19,49.09,-0.10,6.4
6,717,36.55,36.57,0.02,3.5
4,826,32.41,31.41,-1.00,2.9
7,695,30.94,30.19,-0.75,2.3


## Refit on full data and write the submission

Three things change at refit time:

1. **Archetype clusterer is re-fit on all training rows** so cluster
   boundaries reflect the maximum information we have. The test
   archetype assignment is then recomputed from the same clusterer.
2. **Sample weights** are re-estimated on the full data — 2022 (now in
   the training mix) gets the highest weight.
3. **Boosting budget** = `1.2 × best_iteration` from the 2022-fold model,
   to account for having ~30% more training data. (Crude but cheap;
   a proper approach would use a small chronological holdout inside
   2022 to re-tune.)

In [9]:
final_clusterer = ArchetypeClusterer(k=8).fit(train)
train['archetype'] = final_clusterer.predict(train)
test['archetype'] = final_clusterer.predict(test)

X_full = build_extended_features(train)
X_test = build_extended_features(test)
assert list(X_full.columns) == list(X_test.columns)
for c in CAT_FEATURES:
    X_full[c] = X_full[c].astype('int32', errors='ignore')
    X_test[c] = X_test[c].astype('int32', errors='ignore')

y_full_clipped, lo_full, hi_full = clip_target_train_only(y_full, 1.0, 99.0)
full_years = train['start_year'].values
w_full = (full_years - 2019 + 1).astype(float)
w_full = w_full / w_full.mean()

FINAL_ROUNDS = max(50, int(lgb_model.best_iteration * 1.2))
print(f'final model boosting rounds: {FINAL_ROUNDS}')

dfull = lgb.Dataset(X_full, label=y_full_clipped, weight=w_full,
                    categorical_feature=CAT_FEATURES, free_raw_data=False)
final_lgb = lgb.train(PARAMS, dfull, num_boost_round=FINAL_ROUNDS,
                      callbacks=[lgb.log_evaluation(0)])

test_lgb = final_lgb.predict(X_test)

# Final shrunk means use the FULL train (now includes 2022 returns)
final_arc_shrunk = shrunk_archetype_means(train['archetype'], y_full_clipped, prior=200.0)
test_shrunk = test['archetype'].map(final_arc_shrunk).values

test_blend = ALPHA * test_lgb + (1 - ALPHA) * test_shrunk

print(f'\ntest_lgb     mean={test_lgb.mean():.2f}  std={test_lgb.std():.2f}')
print(f'test_shrunk  mean={test_shrunk.mean():.2f}  std={test_shrunk.std():.2f}')
print(f'test_blend   mean={test_blend.mean():.2f}  std={test_blend.std():.2f}  '
      f'(alpha={ALPHA:.2f})')

final model boosting rounds: 50



test_lgb     mean=7.12  std=11.04
test_shrunk  mean=14.45  std=6.53
test_blend   mean=12.25  std=6.11  (alpha=0.30)


In [10]:
submission = sample_submission.copy()
submission['return_pct'] = test_blend
out_path = SUBMISSION_DIR / 'lgb_extended_blend_v2.csv'
submission.to_csv(out_path, index=False)

assert len(submission) == len(sample_submission)
assert list(submission.columns) == ['id', 'return_pct']
print(f'wrote {out_path}')
display(submission.head())
print('rows:', len(submission))

wrote D:\Documents\Programms\kaggle-competition-stock-return-fundamentals\submissions\lgb_extended_blend_v2.csv


,id,return_pct
0,0,4.418534
1,1,9.647215
2,2,10.530309
3,3,5.009763
4,4,18.985370


rows: 8520


## How to read the results

- **`rmse_novel` is the number that matters.** Test = 100% novel tickers,
  and the model has no way to lean on firm-identity memorisation there.
  If the blend's `rmse_novel` beats both LightGBM-alone and shrunk-mean-
  alone, the blend is doing real work.
- The **new-feature gain share** in feature importance tells us whether
  the 31 added features earned their slot. If it is < ~10% the extension
  didn't help; if > ~30% the extension is doing most of the work.
- The `(sector × year)` and `(archetype × year)` z-scores let trees split
  on "cheap for its cohort" rather than "cheap absolutely". That changes
  what the model picks up across regime shifts.

## Next experiments

- **LambdaRank**: replace pointwise Huber with rank loss on within-date
  pairs. Likely the biggest single remaining gain (notebook 06).
- **Pseudo-labeling**: predict test, take the most confident 30%, add as
  soft labels with weight 0.3, retrain. Mitigates the regime gap.
- **Multi-seed bag**: average 5-10 LightGBMs trained with different
  random seeds — typically 0.05-0.10 RMSE for free.
- **CatBoost in the blend**: different splitting heuristics produce
  uncorrelated errors, which is exactly what blending rewards.